# FastAPI Assignment 3 – Product Inventory API

## Objective
The objective of this assignment is to build a REST API using FastAPI that performs CRUD operations on a product inventory system.

This assignment demonstrates how FastAPI can be used to create high-performance APIs with clear endpoint structures.

## Technologies Used
- Python
- FastAPI
- Uvicorn
- Swagger UI

## Concepts Used

### 1. REST API
A REST API allows communication between a client and server using HTTP methods.

### 2. CRUD Operations
CRUD operations represent the basic functions used in databases and APIs.

Create → Add new data  
Read → Retrieve existing data  
Update → Modify existing data  
Delete → Remove data

### 3. HTTP Methods Used

- GET: Used to retrieve product information.

- POST: Used to create a new product in the inventory.

- PUT: Used to update product details such as price or stock status.

- DELETE: Used to remove a product from the inventory.

- Path Order: FastAPI matches routes from top to bottom. Specific routes like /products/audit must be placed above dynamic routes like /products/{product_id} to avoid being captured as a variable.

# **Tools Used**
- Python
- FastAPI
- Uvicorn
- Swagger UI

# **Setup & Initial Data**

In [5]:
from fastapi import FastAPI, Response, status, Query
from pydantic import BaseModel
from typing import Optional

app = FastAPI()

# Initial Database Simulation
products = [
    {"id": 1, "name": "Wireless Mouse", "price": 499, "category": "Electronics", "in_stock": True},
    {"id": 2, "name": "Mechanical Keyboard", "price": 1999, "category": "Electronics", "in_stock": True},
    {"id": 3, "name": "USB Hub", "price": 799, "category": "Electronics", "in_stock": False},
    {"id": 4, "name": "Pen Set", "price": 99, "category": "Stationery", "in_stock": True},
]

class NewProduct(BaseModel):
    name: str
    price: int
    category: str
    in_stock: bool = True

def find_product(product_id: int):
    return next((p for p in products if p["id"] == product_id), None)

# **Q1 - Create Products (POST)**

In [7]:
@app.post("/products", status_code=status.HTTP_201_CREATED)
def add_product(new_item: NewProduct, response: Response):
    # Check for duplicates
    if any(p['name'].lower() == new_item.name.lower() for p in products):
        response.status_code = status.HTTP_400_BAD_REQUEST
        return {"error": f"Product '{new_item.name}' already exists"}
    
    # Auto-generate ID
    new_id = max(p['id'] for p in products) + 1
    product_dict = new_item.dict()
    product_dict['id'] = new_id
    products.append(product_dict)
    
    return {"message": "Product added", "product": product_dict}

# **Question 2 (PUT - Update Product)**

In [9]:
@app.put("/products/{product_id}")
def update_product(product_id: int, price: Optional[int] = None, in_stock: Optional[bool] = None):
    product = find_product(product_id)
    if not product:
        return {"error": "Product not found"}
    
    if price is not None:
        product['price'] = price
    if in_stock is not None:
        product['in_stock'] = in_stock
    return product

# **Question 3 — DELETE (Remove Product)**

In [11]:
@app.delete("/products/{product_id}")
def delete_product(product_id: int, response: Response):
    product = find_product(product_id)
    if not product:
        response.status_code = status.HTTP_404_NOT_FOUND
        return {"error": "Product not found"}
    products.remove(product)
    return {"message": f"Product '{product['name']}' deleted"}

# **Question 4 — Full CRUD Lifecycle Workflow**

### Theory:
- This section demonstrates a complete API lifecycle test, often referred to as an end-to-end (E2E) workflow. In this scenario, we simulate the lifecycle of a product—from its initial entry into the system to its final removal. We will add a "Smart Watch," verify its existence, update its details due to a pricing error, and finally delete it when the product launch is canceled. This test ensures that the Create, Read, Update, and Delete (CRUD) operations are working together seamlessly and maintaining data integrity.

### Workflow Execution Steps (For Swagger/Documentation):

- Since this is a workflow test, you do not need to write new Python logic. Instead, you must execute the existing endpoints in the following sequence and capture the output for each step:

# Step 1: Create (POST /products)

Action: Add a new product named "Smart Watch".

Payload: {"name": "Smart Watch", "price": 3999, "category": "Electronics", "in_stock": false}

Expected Output: A 201 Created status code and a JSON response showing the assigned id.

# Step 2: Verify ID (GET /products)

- Action: Fetch the full list of products to confirm the "Smart Watch" is present.

- Observation: Note the unique id assigned to the Smart Watch (e.g., id: 5).

# Step 3: Update Price (PUT /products/{id})

- Action: Update the price of the Smart Watch to correct a pricing error.

- Parameters: Set product_id to the one noted in Step 2 and update price to 3499.

- Expected Output: JSON response showing the updated price.

# Step 4: Fetch Updated Data (GET /products/{id})

- Action: Retrieve the specific product using its ID.

- Validation: Ensure the response reflects the new price of 3499.

# Step 5: Remove Product (DELETE /products/{id})

- Action: Delete the Smart Watch from the database due to a canceled launch.

- Expected Output: A success message confirming the deletion.

# Step 6: Final Audit (GET /products)

Action: Fetch all products again.

Validation: Confirm the "Smart Watch" is no longer in the list and the total count has returned to its previous state.

# **Question 5 — GET (Inventory Audit)**

In [13]:
@app.get('/products/audit')
def product_audit():
    in_stock_list = [p for p in products if p['in_stock']]
    out_stock_list = [p for p in products if not p['in_stock']]
    stock_value = sum(p['price'] * 10 for p in in_stock_list)
    priciest = max(products, key=lambda p: p['price'])
    return {
        'total_products': len(products),
        'in_stock_count': len(in_stock_list),
        'out_of_stock_names': [p['name'] for p in out_stock_list],
        'total_stock_value': stock_value,
        'most_expensive': {'name': priciest['name'], 'price': priciest['price']},
    }

# **Bonus — PUT (Bulk Category Discount)**

In [37]:
# @app.put('/products/discount')
# def bulk_discount(category: str, discount_percent: int = Query(..., ge=1, le=99)):
#     updated = []
#     for p in products:
#         if p['category'].lower() == category.lower():
#             p['price'] = int(p['price'] * (1 - discount_percent / 100))
#             updated.append(p)
#     if not updated:
#         return {'message': f'No products found in category: {category}'}
#     return {'message': f'Discount applied to {len(updated)} products'}

# --- 1. Sabse Pehle Static Routes (Fixed Names) ---

@app.get("/products/audit")
def product_audit():
    # ... audit code ...
    pass

# DISCOUNT WALA CODE IDHAR RAKHEIN (Dynamic route se upar)
@app.put('/products/discount')
def bulk_discount(category: str, discount_percent: int = Query(..., ge=1, le=99)):
    updated = []
    for p in products:
        if p['category'].lower() == category.lower():
            p['price'] = int(p['price'] * (1 - discount_percent / 100))
            updated.append(p)
    if not updated:
        return {'message': f'No products found in category: {category}'}
    return {'message': f'Discount applied to {len(updated)} products'}


# --- 2. Ab Niche Rakhein Dynamic Routes (Variable {product_id} wale) ---

@app.get("/products/{product_id}")
def get_product(product_id: int, response: Response):
    product = find_product(product_id)
    if not product:
        response.status_code = 404
        return {"error": "Product not found"}
    return product

@app.put("/products/{product_id}")
def update_product(product_id: int, price: int = None):
    # ... update code ...
    pass


# **Server Running Code**

In [41]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8008)

# Start server in a separate thread
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

INFO:     Started server process [2348]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8008 (Press CTRL+C to quit)
